# Font Identifier - Training Notebook

This notebook handles the end-to-end training of the Font Identifier model using EfficientNet-B0. It includes a robust checkpointing system to handle Colab disconnects.

## Step 1 — Verify GPU Access

In [ ]:
!nvidia-smi

## Step 2 — Mount Google Drive
All data and checkpoints will be stored in Drive to survive session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/font-identifier'
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/dataset', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/fonts', exist_ok=True)

print("Drive mounted and folders ready.")

## Step 3 — Clone Repository

In [ ]:
import os
if not os.path.exists('/content/fontidentifier'):
  !git clone https://github.com/jtheanonymous1707-wq/fontidentifier.git /content/fontidentifier
else:
  %cd /content/fontidentifier
  !git pull

%cd /content/fontidentifier/model
!ls -la

## Step 4 — Install Dependencies

In [ ]:
!pip install -q timm supabase python-dotenv
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

## Step 5 — Set Environment Variables

In [ ]:
import os
os.environ['GOOGLE_FONTS_API_KEY'] = 'your_api_key'
os.environ['SUPABASE_URL']         = 'your_supabase_url'
os.environ['SUPABASE_SERVICE_KEY'] = 'your_service_key'
print("Secrets set (locally in memory).")

## Step 6 & 7 — Extract Dataset from Drive
Instead of downloading and generating images in every session, we unzip the pre-generated files from Drive.

In [ ]:
import zipfile
import os
import glob

print("🚀 Starting extraction from Drive...")

DATA_ZIP = f'{DRIVE_DIR}/font_data.zip'

if os.path.exists(DATA_ZIP):
    print(f"📦 Found archive: {DATA_ZIP} ({os.path.getsize(DATA_ZIP)/1024/1024:.2f} MB)")
    print("⌛ Unzipping... this takes ~2-3 minutes.")
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        # Robust extraction: handles any path structure in the zip
        z.extractall('/content/')
    print("✅ Unzip complete.")
else:
    print(f"❌ Error: {DATA_ZIP} not found! Did you upload it to the 'font-identifier' folder correctly?")

DATASET_DIR = '/content/data/dataset'
FONTS_DIR   = '/content/data/fonts'

print("\n--- Verification ---")
if os.path.exists(DATASET_DIR):
    classes = os.listdir(DATASET_DIR)
    print(f"✅ Dataset: {len(classes)} font classes found.")
else:
    print(f"❌ Dataset folder NOT found at {DATASET_DIR}")

if os.path.exists(FONTS_DIR):
    fonts = glob.glob(FONTS_DIR + '/*.*')
    print(f"✅ Fonts: {len(fonts)} files found.")
else:
    print(f"❌ Fonts folder NOT found at {FONTS_DIR}")

if not (os.path.exists(DATASET_DIR) and os.path.exists(FONTS_DIR)):
    print("\n🔍 Debug info: Listing contents of /content/data (if it exists):")
    if os.path.exists('/content/data'):
        print(os.listdir('/content/data'))
    else:
        print("/content/data does not exist.")

## Step 8 — Start Training
This loop includes the explicit auto-resume logic.

In [ ]:
import train
train.DATASET_DIR = '/content/data/dataset'
train.DRIVE_DIR = f'{DRIVE_DIR}/checkpoints'
train.CHECKPOINT_PATH = os.path.join(train.DRIVE_DIR, "latest.pt")
train.BEST_MODEL_PATH = os.path.join(train.DRIVE_DIR, "best_model.pt")

train.train()

## Step 9 — Export Final Model

In [ ]:
import torch
from train import FontNet
import json

device = torch.device('cuda')
# Load best checkpoint to get num_classes
best_ckpt = torch.load(train.BEST_MODEL_PATH, map_location=device)
num_classes = best_ckpt['num_classes']

model = FontNet(num_classes=num_classes).to(device)
model.load_state_dict(best_ckpt['model_state'])
model.eval()

model_cpu = model.cpu()
model_cpu.eval()

scripted = torch.jit.script(model_cpu)
scripted.save(f'{DRIVE_DIR}/font_model_scripted.pt')
print(f"Final model saved to {DRIVE_DIR}/font_model_scripted.pt")